In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import os
import requests
import pandas as pd
import numpy as np

In [2]:
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
TEAM_ID = os.environ.get("TEAM_ID", "team_30")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q: str) -> pd.DataFrame:
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=120)
    r.raise_for_status()
    return pd.DataFrame(r.json().get("data", []))

def get_live_catalog_and_trends() -> pd.DataFrame:
    """Retorna el catàleg vigent amb preus/estoc actualitzats i mètriques de l'última hora."""
    query = """
    WITH latest_updates AS (
        SELECT
            product_id::String AS product_id,
            argMax(category::String, ts::DateTime64(3)) AS category,
            argMax(subcategory::String, ts::DateTime64(3)) AS subcategory,
            argMax(brand::String, ts::DateTime64(3)) AS brand,
            argMax(price_eur::Float64, ts::DateTime64(3)) AS price_eur,
            argMax(description::String, ts::DateTime64(3)) AS description,
            argMax(is_active::Bool, ts::DateTime64(3)) AS is_active,
            argMax(change::String, ts::DateTime64(3)) AS last_change
        FROM shop.catalog_updates
        GROUP BY product_id
    ),
    merged_catalog AS (
        SELECT
            COALESCE(u.product_id, c.product_id::String) AS product_id,
            COALESCE(u.category, c.category::String) AS category,
            COALESCE(u.subcategory, c.subcategory::String) AS subcategory,
            COALESCE(u.brand, c.brand::String) AS brand,
            COALESCE(u.price_eur, c.price_eur::Float64) AS price_eur,
            c.price_eur::Float64 AS initial_price,
            COALESCE(u.description, c.description::String) AS description,
            COALESCE(u.is_active, c.is_active::Bool) AS is_active,
            COALESCE(u.last_change, 'initial') AS last_change
        FROM shop.catalog c
        FULL OUTER JOIN latest_updates u ON c.product_id::String = u.product_id
    ),
    recent_trends AS (
        SELECT
            product_id::String AS product_id,
            countIf(event_type::String = 'view') AS views_1h,
            countIf(event_type::String = 'add_to_cart') AS carts_1h,
            countIf(event_type::String = 'purchase') AS purchases_1h
        FROM shop.events
        WHERE ts::DateTime64(3) >= now() - INTERVAL 1 HOUR
        GROUP BY product_id
    )
    SELECT
        m.*,
        COALESCE(t.views_1h, 0) AS views_1h,
        COALESCE(t.carts_1h, 0) AS carts_1h,
        COALESCE(t.purchases_1h, 0) AS purchases_1h
    FROM merged_catalog m
    LEFT JOIN recent_trends t ON m.product_id = t.product_id
    """
    return sql(query)

In [3]:
class ItemTower(nn.Module):
    def __init__(self, n_cats: int, n_subcats: int, n_brands: int,
                 text_dim: int = 32, num_dim: int = 6, emb_dim: int = 64):
        super().__init__()
        self.cat_emb = nn.Embedding(n_cats + 1, 16, padding_idx=0)
        self.subcat_emb = nn.Embedding(n_subcats + 1, 24, padding_idx=0)
        self.brand_emb = nn.Embedding(n_brands + 1, 16, padding_idx=0)
        
        in_dim = 16 + 24 + 16 + text_dim + num_dim
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, emb_dim)
        )

    def forward(self, cat_idx, subcat_idx, brand_idx, text_vec, num_feats):
        x = torch.cat([
            self.cat_emb(cat_idx),
            self.subcat_emb(subcat_idx),
            self.brand_emb(brand_idx),
            text_vec,
            num_feats
        ], dim=-1)
        return F.normalize(self.mlp(x), p=2, dim=-1)


class UserTower(nn.Module):
    def __init__(self, item_tower: ItemTower, n_devices: int, n_regions: int,
                 n_pages: int, emb_dim: int = 64):
        super().__init__()
        self.item_tower = item_tower
        self.dev_emb = nn.Embedding(n_devices + 1, 8, padding_idx=0)
        self.reg_emb = nn.Embedding(n_regions + 1, 12, padding_idx=0)
        self.page_emb = nn.Embedding(n_pages + 1, 8, padding_idx=0)
        
        # Pesos d'atenció per ponderar quins productes recents importen més
        self.hist_attn = nn.Linear(emb_dim + 1, 1)
        
        in_dim = emb_dim + 8 + 12 + 8 + 2  # +2 per is_new_user i log(n_views)
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, emb_dim)
        )

    def forward(self, hist_item_embs, hist_dt, hist_mask,
                dev_idx, reg_idx, page_idx, user_num_feats):
        """
        hist_item_embs: (B, L, emb_dim) embeddings precalculats dels recent_views
        hist_dt:        (B, L, 1) segons transcorreguts (log1p)
        hist_mask:      (B, L) booleà (True = posició vàlida)
        """
        attn_in = torch.cat([hist_item_embs, hist_dt], dim=-1)
        scores = self.hist_attn(attn_in).squeeze(-1)  # (B, L)
        scores = scores.masked_fill(~hist_mask, -1e9)
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)  # (B, L, 1)
        
        # Si l'usuari no té recent_views (ex: home page nou usuari), vector zero
        has_hist = hist_mask.any(dim=-1, keepdim=True).float()
        pooled_hist = (hist_item_embs * weights).sum(dim=1) * has_hist

        x = torch.cat([
            pooled_hist,
            self.dev_emb(dev_idx),
            self.reg_emb(reg_idx),
            self.page_emb(page_idx),
            user_num_feats
        ], dim=-1)
        return F.normalize(self.mlp(x), p=2, dim=-1)


class TwoTowerRecommender(nn.Module):
    def __init__(self, item_tower: ItemTower, user_tower: UserTower, temperature: float = 0.07):
        super().__init__()
        self.item_tower = item_tower
        self.user_tower = user_tower
        self.temperature = temperature

    def compute_loss(self, user_embs, pos_item_embs, sample_weights, item_log_probs=None):
        """
        InfoNCE / Sampled Softmax amb correcció de popularitat (Log-Q) i pesos per conversió.
        """
        logits = torch.matmul(user_embs, pos_item_embs.T) / self.temperature
        if item_log_probs is not None:
            # Correcció Log-Q per evitar que el model només recomani best-sellers
            logits = logits - item_log_probs.unsqueeze(0)
        
        targets = torch.arange(user_embs.size(0), device=user_embs.device)
        loss_per_sample = F.cross_entropy(logits, targets, reduction="none")
        return (loss_per_sample * sample_weights).mean()

In [4]:
class FeatureEncoder:
    def __init__(self, text_dim: int = 32):
        self.cat_map = {}
        self.subcat_map = {}
        self.brand_map = {}
        self.dev_map = {"web": 1, "ios": 2, "android": 3}
        self.reg_map = {}
        self.page_map = {"home": 1, "product": 2, "category": 3}
        self.tfidf = TfidfVectorizer(max_features=2000, stop_words="english")
        self.svd = TruncatedSVD(n_components=text_dim, random_state=42)
        self.fitted = False

    def fit_catalog(self, df: pd.DataFrame):
        self.cat_map = {v: i + 1 for i, v in enumerate(df["category"].dropna().unique())}
        self.subcat_map = {v: i + 1 for i, v in enumerate(df["subcategory"].dropna().unique())}
        self.brand_map = {v: i + 1 for i, v in enumerate(df["brand"].dropna().unique())}
        tfidf_mat = self.tfidf.fit_transform(df["description"].fillna(""))
        self.svd.fit(tfidf_mat)
        self.fitted = True

    def encode_items(self, df: pd.DataFrame) -> dict:
        cat_idx = torch.tensor([self.cat_map.get(x, 0) for x in df["category"]], dtype=torch.long)
        subcat_idx = torch.tensor([self.subcat_map.get(x, 0) for x in df["subcategory"]], dtype=torch.long)
        brand_idx = torch.tensor([self.brand_map.get(x, 0) for x in df["brand"]], dtype=torch.long)
        
        text_vec = torch.tensor(
            self.svd.transform(self.tfidf.transform(df["description"].fillna(""))),
            dtype=torch.float32
        )
        
        price = df["price_eur"].fillna(20.0).values
        init_price = df["initial_price"].fillna(price).values
        price_ratio = price / np.maximum(init_price, 1.0)
        
        num_feats = np.stack([
            np.log1p(np.maximum(price, 0.0)) / 5.0,
            np.clip(price_ratio, 0.2, 2.0),
            (df["last_change"] == "new_product").astype(float).values,
            np.log1p(df["views_1h"].values) / 5.0,
            np.log1p(df["carts_1h"].values) / 3.0,
            np.log1p(df["purchases_1h"].values) / 2.0,
        ], axis=-1)
        
        return {
            "cat_idx": cat_idx,
            "subcat_idx": subcat_idx,
            "brand_idx": brand_idx,
            "text_vec": text_vec,
            "num_feats": torch.tensor(num_feats, dtype=torch.float32),
        }

In [5]:
import time
import threading
from datetime import datetime, timezone

class LiveEngine:
    def __init__(self, model: TwoTowerRecommender, encoder: FeatureEncoder):
        self.model = model
        self.model.eval()
        self.encoder = encoder
        self.lock = threading.Lock()
        
        self.item_ids = []
        self.id_to_idx = {}
        self.active_mask = None
        self.item_embs = None       # (N_items, emb_dim)
        self.trend_bonus = None     # (N_items,)
        self.ucb_bonus = None       # (N_items,)
        self.user_bought = {}       # user_id -> set(product_id)

    def refresh_state(self):
        """Executat cada 60s en background: actualitza catàleg, tendències, compres i feedback."""
        try:
            cat_df = get_live_catalog_and_trends()
            tensors = self.encoder.encode_items(cat_df)
            with torch.no_grad():
                embs = self.model.item_tower(
                    tensors["cat_idx"], tensors["subcat_idx"], tensors["brand_idx"],
                    tensors["text_vec"], tensors["num_feats"]
                )
            
            # Petit impuls a productes amb alta conversió recent (trend signal)
            conv_rate_1h = (cat_df["purchases_1h"].values + 0.1) / (cat_df["views_1h"].values + 10.0)
            trend_bonus = torch.tensor(np.clip(conv_rate_1h * 0.5, 0.0, 0.15), dtype=torch.float32)

            # Bonus d'exploració/explotació basat en el nostre propi feedback (team_30.feedback)
            fb_df = sql(f"""
                SELECT
                    arrayJoin(shown) AS product_id,
                    count() AS impressions,
                    countIf(purchased::String = product_id) AS buys
                FROM {TEAM_ID}.feedback
                WHERE request_id != '__init__' AND served_by = 'team'
                GROUP BY product_id
            """)
            ucb = np.zeros(len(cat_df), dtype=np.float32)
            id_map = {pid: i for i, pid in enumerate(cat_df["product_id"])}
            if not fb_df.empty:
                total_imp = fb_df["impressions"].sum() + 1
                for row in fb_df.itertuples():
                    idx = id_map.get(row.product_id)
                    if idx is not None:
                        emp_cr = row.buys / (row.impressions + 5.0)
                        explore = 0.05 * np.sqrt(np.log(total_imp) / (row.impressions + 1.0))
                        ucb[idx] = np.clip(emp_cr + explore, 0.0, 0.20)

            # Productes ja comprats per cada usuari per no repetir-los
            bought_df = sql("""
                SELECT user_id::String AS user_id, groupArray(DISTINCT product_id::String) AS items
                FROM (
                    SELECT user_id, product_id FROM shop.history_events WHERE event_type = 'purchase'
                    UNION ALL
                    SELECT user_id, product_id FROM shop.events WHERE event_type = 'purchase'
                )
                GROUP BY user_id
            """)
            bought_dict = {r.user_id: set(r.items) for r in bought_df.itertuples()}

            with self.lock:
                self.item_ids = cat_df["product_id"].tolist()
                self.id_to_idx = id_map
                self.active_mask = torch.tensor(cat_df["is_active"].values, dtype=torch.bool)
                self.item_embs = embs
                self.trend_bonus = trend_bonus
                self.ucb_bonus = torch.tensor(ucb, dtype=torch.float32)
                self.user_bought = bought_dict
        except Exception as e:
            print(f"[Refresh warning] {e}")

    def recommend_batch(self, requests_df: pd.DataFrame, max_hist: int = 10) -> list[dict]:
        """Prediu el top-10 per a un batch de peticions en < 5ms."""
        with self.lock:
            item_embs = self.item_embs
            active_mask = self.active_mask
            trend_bonus = self.trend_bonus
            ucb_bonus = self.ucb_bonus
            id_to_idx = self.id_to_idx
            item_ids = self.item_ids
            user_bought = self.user_bought

        B = len(requests_df)
        emb_dim = item_embs.size(1)
        hist_embs = torch.zeros((B, max_hist, emb_dim), dtype=torch.float32)
        hist_dt = torch.zeros((B, max_hist, 1), dtype=torch.float32)
        hist_mask = torch.zeros((B, max_hist), dtype=torch.bool)

        dev_idx, reg_idx, page_idx, num_feats = [], [], [], []

        for b, row in enumerate(requests_df.itertuples()):
            ctx = row.context if isinstance(row.context, dict) else {}
            dev_idx.append(self.encoder.dev_map.get(ctx.get("device"), 0))
            reg_str = ctx.get("region", "")
            if reg_str not in self.encoder.reg_map:
                self.encoder.reg_map[reg_str] = min(len(self.encoder.reg_map) + 1, 25)
            reg_idx.append(self.encoder.reg_map.get(reg_str, 0))
            page_idx.append(self.encoder.page_map.get(ctx.get("page"), 0))

            rviews = row.recent_views or []
            recent_slice = rviews[-max_hist:]
            req_ts = pd.Timestamp(row.ts)

            for l, v in enumerate(recent_slice):
                pid = v.get("product_id")
                idx = id_to_idx.get(pid)
                if idx is not None:
                    hist_embs[b, l] = item_embs[idx]
                    hist_mask[b, l] = True
                    dt_sec = max((req_ts - pd.Timestamp(v.get("ts", row.ts))).total_seconds(), 0.0)
                    hist_dt[b, l, 0] = np.log1p(dt_sec) / 10.0

            num_feats.append([float(bool(row.is_new_user)), np.log1p(len(rviews)) / 3.0])

        with torch.no_grad():
            u_embs = self.model.user_tower(
                hist_embs, hist_dt, hist_mask,
                torch.tensor(dev_idx, dtype=torch.long),
                torch.tensor(reg_idx, dtype=torch.long),
                torch.tensor(page_idx, dtype=torch.long),
                torch.tensor(num_feats, dtype=torch.float32)
            )
            # Puntuació base + senyals en viu
            scores = torch.matmul(u_embs, item_embs.T) + trend_bonus.unsqueeze(0) + ucb_bonus.unsqueeze(0)
            # Emmascarem productes fora d'estoc
            scores[:, ~active_mask] = -1e9

        submissions = []
        for b, row in enumerate(requests_df.itertuples()):
            row_scores = scores[b].clone()
            # Penalitzem productes ja comprats per aquest usuari
            already_bought = user_bought.get(row.user_id, set())
            for pid in already_bought:
                idx = id_to_idx.get(pid)
                if idx is not None:
                    row_scores[idx] -= 2.0

            # Opcional: si estem en pàgina de producte, no recomanar exactament el mateix producte que mira ara
            if row.recent_views:
                last_pid = row.recent_views[-1].get("product_id")
                last_idx = id_to_idx.get(last_pid)
                if last_idx is not None:
                    row_scores[last_idx] -= 0.5

            top10_idx = torch.topk(row_scores, k=10).indices.tolist()
            submissions.append({
                "request_id": row.request_id,
                "items": [item_ids[i] for i in top10_idx],
                "model_version": "v1-two-tower-adaptive"
            })
        return submissions

In [6]:
def run_live_loop(engine: LiveEngine):
    engine.refresh_state()
    
    # Fil en segon pla que refresca catàleg i feedback cada 60 segons
    def background_updater():
        while True:
            time.sleep(60)
            engine.refresh_state()

    threading.Thread(target=background_updater, daemon=True).start()
    
    seen_requests = set()
    last_ts = (datetime.now(timezone.utc) - pd.Timedelta(seconds=10)).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

    while True:
        t0 = time.time()
        try:
            reqs = sql(f"""
                SELECT request_id, ts, user_id, is_new_user, context, recent_views
                FROM shop.requests
                WHERE ts::DateTime64(3) >= '{last_ts}'
                ORDER BY ts::DateTime64(3) ASC
                LIMIT 100
            """)
            if not reqs.empty:
                reqs = reqs[~reqs["request_id"].isin(seen_requests)]
                if not reqs.empty:
                    subs = engine.recommend_batch(reqs)
                    # Inserció en batch a team_30.submissions
                    res = http.post(
                        f"{API}/v1/tables/submissions",
                        params={"database": TEAM_ID},
                        json=subs,
                        timeout=2.0
                    )
                    res.raise_for_status()
                    
                    seen_requests.update(reqs["request_id"].tolist())
                    if len(seen_requests) > 10000:
                        seen_requests = set(list(seen_requests)[-5000:])
                    last_ts = str(reqs["ts"].max())[:23]
        except Exception as e:
            print(f"[Loop error] {e}")

        elapsed = time.time() - t0
        time.sleep(max(0.05, 0.3 - elapsed))